# Prism Eval

Runs the Prism Recipe against baseline on nanoGPT Shakespeare across multiple
seeds and produces the **Prism Score** — how many fewer steps Prism needs to
reach baseline quality.

Score 1.0 = no benefit. Higher = better. There is no expected value: whatever
this prints is the result.

**Read the score against `baseline_best`.** The Prism Score is a ratio, so a
weak baseline inflates it without the method improving. A score marked
*lower bound* means the target was hit at the first eval and the true crossing
is unresolved.

### Built to survive flaky Colab

This run is **stepwise and resumable**. Each expensive stage — per seed:
teacher, baseline, method — is banked to disk (`.prism_runs/`) the moment it
finishes, and a full artifact is written to `results/` after every seed
(flagged `partial` until the last). So an interruption costs you at most the
one stage that was in flight, never the finished ones.

Two things make it robust to a dropped runtime:

1. **The eval runs detached.** The launch cell starts it in its own session and
   just *tails the log*. Stopping the cell — or a websocket disconnect that
   turns the spinner back into a play button — does **not** stop the run. Re-run
   the cell and it **re-attaches** to the still-live process; it never starts a
   second one.
2. **A relaunch resumes, it doesn't restart.** If the process really did die,
   re-running the launch cell starts a fresh eval that skips every stage already
   on disk and picks up where it stopped. A finished 110-minute baseline is not
   recomputed.

The one thing on-disk resume can't survive is a full Colab **factory reset**
(which wipes `/content`). Against that, only committing `results/*.json` helps —
so run the artifact cell and commit as soon as it lands.

**A number without a committed artifact is not a result.** Save this notebook
**with outputs**, then commit the file the artifact cell downloads.

Runtime (measured): ~4 h per seed on a T4, so ~12 h for 3 seeds. The model is
small (10.65M params) and underutilizes a large GPU, so an A100 is faster but
not by an order of magnitude — budget a few hours. A free T4 will likely
disconnect before finishing; the resume machinery above is what makes that
survivable, but paid runtime still finishes in one sitting.

---
*[github.com/timepointai/nanogpt-prism-shakespeare](https://github.com/timepointai/nanogpt-prism-shakespeare) · [Sean McDonald](https://x.com/seanmcdonaldxyz)*

## Transfer improvements (opt-in)

The launch cell runs the committed recipe by default. To test a lever, set
`FLAGS` in that cell. Every default is unchanged, and each non-default knob gets
its own run key — so it produces a separate `results/*.json` and never
false-resumes onto a plain-recipe result. Full menu: `IMPROVEMENTS.md`.

Direction transfer (the differentiated headroom — the singular vectors):

- `--align_mode=grassmann` — pair directions by geometry, not by index
- `--align_topk=32` — transfer only the leading k directions, keep the tail fresh
- `--align_mode=subspace` — set per-matrix strength from student↔teacher distance
- `--align_spec=attention:0.9,ffn_down:0.5` · `--align_depth_gamma=0.5`

Spectral imprint: `--n_dct=16` (truer spectrum) · `--per_layer` (per-matrix).

Representational transfer: `--cka=0.1 --cka_layers=2,4` (match teacher activations).

**The structure-vs-content test.** `overlap 0.0` is still same-corpus (small
token-JS). `--far_corpus=data/far.txt` swaps the student's non-shared blocks for
a genuinely different corpus, so token-JS is large. Read the recipe advantage
against the `tok-JS` column: advantage that persists as token-JS grows is
structural; advantage that vanishes was content.


In [ ]:
# Optional: validate the new transfer math offline (seconds, no training).
# Run after the launch cell has cloned the repo. Safe to skip.
import subprocess, sys
REPO = '/content/nanogpt-prism'
print(subprocess.run([sys.executable, 'prism_selftest.py'],
                     cwd=f'{REPO}/src', capture_output=True, text=True).stdout)


In [ ]:
import os, subprocess, sys, time

REPO = '/content/nanogpt-prism'
LOG = '/content/prism_eval.log'
PIDFILE = '/content/prism_eval.pid'
os.chdir('/content')

# Branch to run. 'improvement' carries the opt-in transfer levers (grassmann /
# top-k / subspace / per-layer / CKA / far-corpus). Set to 'master' for the
# committed recipe only.
BRANCH = 'improvement'

# Extra flags appended to the run. Empty = the committed recipe, unchanged.
# Examples (see the "Transfer improvements" cell above):
#   FLAGS = ['--align_mode=grassmann', '--align_topk=32']
#   FLAGS = ['--n_dct=16', '--per_layer']
#   FLAGS = ['--overlap=0.0', '--far_corpus=data/far.txt',
#            '--method_lr=1e-3', '--method_warmup=100']
FLAGS = []

# 1) Get the code WITHOUT wiping resume state. A prior attempt may have banked
#    finished stages under REPO/.prism_runs/ — rmtree would throw that away and
#    we'd recompute 110-minute runs for nothing. Reuse if present; pull code
#    updates (git pull leaves untracked run artifacts alone); clone only if new.
if os.path.isdir(REPO):
    print('Repo already here — reusing it; resume state in .prism_runs/ is kept.')
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH,
        'https://github.com/timepointai/nanogpt-prism-shakespeare.git', REPO],
        check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'tiktoken', 'datasets'], check=True)

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — CPU/MPS, will be slow"}')
print('branch:', BRANCH, '| commit:',
      subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip())
if FLAGS:
    print('extra flags:', ' '.join(FLAGS))


def alive(pid):
    try:
        os.kill(pid, 0); return True
    except Exception:
        return False


# 2) Is an eval already running from an earlier attempt that got disconnected?
#    A websocket drop stops the CELL, not the detached process — so re-running
#    this cell should RE-ATTACH to the live run, never launch a second one.
running = None
if os.path.exists(PIDFILE):
    try:
        pid = int(open(PIDFILE).read().strip())
        if alive(pid):
            running = pid
    except Exception:
        pass

if running:
    print(f'\nEval already running (pid {running}) — it survived the disconnect. '
          f'Re-attaching to its log; nothing relaunched.\n')
else:
    # 3) Launch DETACHED (start_new_session) so a kernel interrupt / websocket
    #    hiccup can't kill it. Output goes to LOG (append), which we then tail.
    #    The script itself resumes from .prism_runs/, so even a fresh launch
    #    skips whatever already finished.
    print('\nLaunching eval — detached, so stopping this cell will NOT stop the run.\n')
    logf = open(LOG, 'ab')
    proc = subprocess.Popen(
        [sys.executable, '-u', 'prism_eval.py',
         '--method=recipe', '--teacher_steps=2000', '--student_steps=5000',
         '--seeds=1337,1338,1339'] + FLAGS,
        cwd=f'{REPO}/src', stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True)
    open(PIDFILE, 'w').write(str(proc.pid))
    running = proc.pid
    time.sleep(2)

# 4) Tail the log live. `--pid` makes tail exit on its own when the eval
#    finishes; `-n +1` replays the whole log so a re-attach shows full history.
#    Stopping THIS cell is safe — the run keeps going; re-run to resume tailing.
print(f'--- tailing {LOG}  (stopping this cell is safe; the run continues) ---\n')
tail = subprocess.Popen(['tail', '-n', '+1', f'--pid={running}', '-f', LOG],
                        stdout=subprocess.PIPE, text=True)
for line in tail.stdout:
    print(line, end='')
tail.wait()

if not alive(running):
    try:
        os.remove(PIDFILE)
    except Exception:
        pass
    print('\n--- eval process has exited. Scroll up for the summary, then run '
          'the artifact cell to print + download the artifact and COMMIT it. ---')

In [ ]:
# Print the artifact (captured in this notebook's saved outputs) and download it.
import json, glob, os

art = sorted(glob.glob(f'{REPO}/results/recipe_*.json'))[-1]
print(json.dumps(json.load(open(art)), indent=2))
print(f'\nCommit this to results/: {os.path.basename(art)}')

try:
    from google.colab import files
    files.download(art)
except Exception as e:
    print(f'(no Colab download: {e})')